# Chapter 8, Exercise 2: Macro-F1 from the Figure 8.8 confusion matrix

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 8, Exercise 2.** Using Python, implement the five steps of Section 8.11 on the four-class confusion matrix printed in Figure 8.8. Extract the true positives, false positives, and false negatives for each dialect, compute the per-class precision and recall, then the macro-averaged F1 score, and check that you recover the accuracy and the macro-F1 the figure reports. Then raise the rare class from fifty test utterances to five hundred, holding its error rate fixed, and explain what happens to each of the two numbers and why.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


## 1. The confusion matrix of Figure 8.8

Rows are true labels, columns are predicted labels (illustrative counts from the book).

In [1]:
import numpy as np, pandas as pd
CLASSES = ["MSA", "Egyptian", "Levantine", "Gulf"]
C = np.array([[470,  15,  10,   5],    # true MSA
              [ 20, 255,  20,   5],    # true Egyptian
              [ 10,  15, 105,  20],    # true Levantine
              [  4,   3,  25,  18]])   # true Gulf  (rare class: 50 utterances)
cm = pd.DataFrame(C, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES])
cm["row total"] = C.sum(axis=1)
cm

,pred MSA,pred Egyptian,pred Levantine,pred Gulf,row total
true MSA,470,15,10,5,500
true Egyptian,20,255,20,5,300
true Levantine,10,15,105,20,150
true Gulf,4,3,25,18,50


## 2. The five steps, for each class

1. TP = diagonal cell; 2. FP = rest of the column; 3. FN = rest of the row; 4. precision = TP/(TP+FP), recall = TP/(TP+FN); 5. F1 = 2PR/(P+R), or 0 when both are 0. Macro-F1 is the unweighted mean of the class F1 values; accuracy is the trace divided by the total.

In [2]:
def macro_f1_report(C, classes):
    rows = []
    for i, c in enumerate(classes):
        TP = C[i, i]
        FP = C[:, i].sum() - TP          # rest of the column
        FN = C[i, :].sum() - TP          # rest of the row
        P = TP / (TP + FP) if TP + FP else 0.0
        R = TP / (TP + FN) if TP + FN else 0.0
        F1 = 2*P*R/(P + R) if P + R else 0.0
        rows.append({"class": c, "support": C[i].sum(), "TP": TP, "FP": FP, "FN": FN,
                     "precision": P, "recall": R, "F1": F1})
    df = pd.DataFrame(rows).set_index("class")
    macro = df["F1"].mean()                       # unweighted mean of the UNROUNDED class F1 values
    acc = np.trace(C) / C.sum()
    return df, macro, acc

df, macro, acc = macro_f1_report(C, CLASSES)
print(df.round(3))
print(f"\nmacro-F1 = {macro:.4f}  (figure reports 0.71)")
print(f"accuracy = {acc:.4f}  (figure reports 0.85)")

           support   TP  FP  FN  precision  recall     F1
class                                                    
MSA            500  470  34  30      0.933    0.94  0.936
Egyptian       300  255  33  45      0.885    0.85  0.867
Levantine      150  105  55  45      0.656    0.70  0.677
Gulf            50   18  30  32      0.375    0.36  0.367

macro-F1 = 0.7121  (figure reports 0.71)
accuracy = 0.8480  (figure reports 0.85)


Both numbers match the figure: accuracy 0.848 (shown as 0.85) and macro-F1 0.712 (shown as 0.71). The Gulf row reproduces the highlighted values TP = 18, FP = 30, FN = 32, precision 0.38, recall 0.36, F1 0.37.

## 3. Raise the rare class from 50 to 500 utterances, error rate held fixed

"Holding its error rate fixed" means the Gulf row keeps the same proportions: 36 % recall (18/50) and the same distribution of its misses over the other classes. We scale the Gulf row by 10 and leave the other rows unchanged.

In [3]:
C2 = C.copy().astype(float)
C2[3, :] = C[3, :] * 10          # Gulf row: 40, 30, 250, 180 (still 36 % correct)
df2, macro2, acc2 = macro_f1_report(C2, CLASSES)
print(df2.round(3))
print(f"\nmacro-F1: {macro:.3f} -> {macro2:.3f}")
print(f"accuracy: {acc:.3f} -> {acc2:.3f}")
print(f"\nGulf precision {df.loc['Gulf','precision']:.2f} -> {df2.loc['Gulf','precision']:.2f} "
      f"(the 30 false positives from other classes are now diluted by 180 true positives)")
print(f"Levantine precision {df.loc['Levantine','precision']:.2f} -> {df2.loc['Levantine','precision']:.2f} "
      f"(250 Gulf utterances now land in the Levantine column)")

           support     TP     FP     FN  precision  recall     F1
class                                                            
MSA          500.0  470.0   70.0   30.0      0.870    0.94  0.904
Egyptian     300.0  255.0   60.0   45.0      0.810    0.85  0.829
Levantine    150.0  105.0  280.0   45.0      0.273    0.70  0.393
Gulf         500.0  180.0   30.0  320.0      0.857    0.36  0.507

macro-F1: 0.712 -> 0.658
accuracy: 0.848 -> 0.697

Gulf precision 0.38 -> 0.86 (the 30 false positives from other classes are now diluted by 180 true positives)
Levantine precision 0.66 -> 0.27 (250 Gulf utterances now land in the Levantine column)


## 4. What happened and why (executed values)

| | Gulf = 50 utterances | Gulf = 500 utterances |
|---|---|---|
| accuracy | 0.848 | 0.697 |
| macro-F1 | 0.712 | 0.658 |
| Gulf precision / recall / F1 | 0.38 / 0.36 / 0.37 | 0.86 / 0.36 / 0.51 |
| Levantine precision / recall / F1 | 0.66 / 0.70 / 0.68 | 0.27 / 0.70 / 0.39 |

* **Accuracy falls by 15 points.** It is a support-weighted count: when a class on which the system is only 36 % correct grows from 5 % to 36 % of the test set, the pooled number moves toward that class's poor rate. Nothing about the system changed; only the composition of the test set did. This is why an accuracy on an imbalanced set flatters the system.
* **Macro-F1 also falls, but for a different reason.** The Gulf class's *weight* in macro-F1 is one quarter before and after, and its recall is fixed by construction, so its own contribution does not shrink; in fact its F1 **rises** (0.37 to 0.51) because its 30 false positives are now diluted by 180 true positives. What drags macro-F1 down is the **Levantine** class: 250 misclassified Gulf utterances now land in its column, its precision collapses from 0.66 to 0.27, and its F1 halves. Macro-F1 is insensitive to how many test utterances a class has, but it is not insensitive to where that class's errors go.

The lesson of Section 8.11: with the rare class small, accuracy (0.85) hides the Gulf problem while macro-F1 (0.71) exposes it. Enlarging the rare class makes accuracy finally reflect the problem, and moves the damage in macro-F1 from the Gulf row to the Levantine column. Report both numbers with the support of every class, and always show the confusion matrix, because the Gulf-to-Levantine confusion (neighbouring varieties, Exercise 1) is the real finding and is invisible in either summary.